# SC4055 Acrobot — 持久保存与续跑版 v2

**不再运行旧版 ABC 单元。** 本 notebook 内含源码，不需要上传 ZIP。
使用免费 CPU runtime 即可。先运行全部；第一个单元需要你批准挂载自己的 Google Drive。
后续先运行测试，再在 Drive 写入读回检查通过后执行 A→B→C。

持久目录：`我的云端硬盘/SC4055_Acrobot_Recovery/experiment_v2`。
每个评估回合立即写入 Drive；CSV 每五回合导出一次。
运行时失联后，重新打开这个 notebook、挂载同一个 Drive、再次运行全部：
完成的训练 seeds 自动跳过，评估从已保存的下一个回合继续。
**一个训练 seed 中途断开仍须重跑该 seed**，不宣称恢复优化器和 replay buffer。
不用删除文件，不用改变共享权限，不消耗队友 GitHub 配额。

旧运行时已丢失的模型无法凭日志复原，所以 A 需要重新计算。
本版本没有用编造数据填补丢失结果。不要同时运行两个本 notebook 会话。


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, subprocess, shutil, base64, hashlib, zipfile, io

drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/SC4055_Acrobot_Recovery')
PERSIST.mkdir(parents=True, exist_ok=True)
SOURCE_B64 = '...'
EXPECTED_SHA256 = '7367e92e04f8d56e86f655abcc6bc99bace2e03dd2f437884ae012340f7d5cbf'

source = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(source).hexdigest() == EXPECTED_SHA256
(PERSIST / 'source_v2.zip').write_bytes(source)
assert hashlib.sha256((PERSIST / 'source_v2.zip').read_bytes()).hexdigest() == EXPECTED_SHA256
ROOT = Path('/content/acrobot_recovery/SC4055-Project')
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    destination = ROOT.parent.resolve()
    for member in archive.infolist():
        target = (destination / member.filename).resolve()
        if ROOT != target and ROOT not in target.parents:
            raise ValueError('Invalid embedded source path')
    archive.extractall(destination)
(PERSIST / 'SOURCE_SHA256.txt').write_text(EXPECTED_SHA256)
print('Drive 源码写入及读回校验通过：', PERSIST)
print('请在 Colab 文件栏确认 drive/MyDrive/SC4055_Acrobot_Recovery/source_v2.zip 可见。')


## 环境与验证（每次新 runtime 都可重复执行）
独立 Python 3.12，固定项目版本，修复 MPLBACKEND 和子进程输出。
测试包括：参数匹配、梯度、训练保存、评估重复执行跳过、协议不匹配拒绝，
以及加速 shots 的概率电路与原电路一致、有限采样方差和联合相关性。
测试失败将停止，不会启动长实验。当前 Windows 无法运行 JAX，新增测试需在这里实际验证。


In [ ]:
def visible_run(command, **kwargs):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, **kwargs)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)
    except BaseException:
        # Stop the child if the user interrupts; avoid an orphan training process.
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        raise
    finally:
        process.stdout.close()

visible_run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.13.7'])
UV = shutil.which('uv')
assert UV, 'uv not on PATH'
ENV = os.environ.copy()
ENV.update(MPLBACKEND='Agg', JAX_PLATFORMS='cpu', PYTHONUNBUFFERED='1',
           UV_PYTHON_INSTALL_DIR='/content/acrobot_python', UV_CACHE_DIR='/content/acrobot_uv_cache')
visible_run([UV, 'python', 'install', '3.12', '--no-bin'], env=ENV)
if not (ROOT / '.venv/bin/python').exists():
    visible_run([UV, 'venv', '--python', '3.12', str(ROOT / '.venv')], env=ENV)
PY = str(ROOT / '.venv/bin/python')
ENV['PATH'] = str(ROOT / '.venv/bin') + os.pathsep + ENV['PATH']
visible_run([UV, 'pip', 'install', '--python', PY, 'swig>=4.2'], env=ENV)
visible_run([UV, 'pip', 'install', '--python', PY, '-e', '.', '--group', 'test'], cwd=ROOT, env=ENV)

def run(*args):
    visible_run([PY, '-u', *map(str, args)], cwd=ROOT, env=ENV)

run('-m', 'pytest', 'tests/sc4055_project/tasks/test_acrobot.py', '-q')
environment = subprocess.check_output([UV, 'pip', 'freeze', '--python', PY], env=ENV, text=True)
(PERSIST / 'environment_v2.txt').write_text(environment)
print('环境和测试已通过。')


## 执行完整 A → B → C，自动保存与跳过

不缩减原计划：A 为三模型各五 seeds、600 回合；B 为五个 VQC、17 条件各100回合；
C 为 depolarizing=0.01、三个 seeds、600回合，并评估 clean 和匹配噪声。

完整耗时尚未可靠测量。进度会每五个评估回合输出，且不再依赖最终 ZIP 下载保住数据。
有限 shots 使用真实联合分布抽样加速，未替换成精确期望值；该抽样实现与旧版 RNG
序列不同，所以本轮使用独立 experiment_v2 目录，不把新旧有限 shots 当成逐位相同结果。

如果中断：重新运行本 notebook。完成的训练和评估自动跳过。若修改了代码或协议，
程序会拒绝混用旧结果，不会静默覆盖。遇到错误请保留完整输出；已写入 Drive 的数据保留。


In [ ]:
OUT = PERSIST / 'experiment_v2'
run('analysis/acrobot/recoverable.py', '--out', OUT, '--stage', 'ABC')
print('ABC 全部完成，原始数据和 checkpoint 已保存在：', OUT)


## 导出（可选；原始结果已经在 Drive）
可在第三单元完成或停止后运行。ZIP 也保存到 Drive，下载失败不会丢失原数据。


In [ ]:
if (PERSIST / 'experiment_v2').exists():
    archive = shutil.make_archive(str(PERSIST / 'Acrobot_v2_results'), 'zip',
                                   root_dir=PERSIST, base_dir='experiment_v2')
    print('结果 ZIP 已保存到 Drive：', archive)
    from google.colab import files
    files.download(archive)
else:
    print('尚无实验输出目录。')
